In [42]:
import torch
from torch import nn
import numpy as np
import requests
import torch_directml
import torch.nn.functional as F

In [43]:
SEQ_LENGTH = 100
BATCH_SIZE = 64
EMBEDDING_DIM = 128
HIDDEN_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
NUM_LAYERS = 2
DEVICE = "cpu"
CLIP = 5

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [44]:
response = requests.get(URL)
text = response.text
text = text[:100000]

chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: i for i, ch in int2char.items()}

encoded_text = np.array([char2int[ch] for ch in text])

VOCAB_SIZE = len(chars)

In [45]:
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr) // batch_size_total
    arr = arr[:n_batches * batch_size_total]

    arr = arr.reshape((batch_size, -1))

    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        y = np.zeros_like(x)

        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]

        yield x, y


batches = get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH)
x_sample, y_sample = next(batches)

In [46]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        lstm_out = lstm_out.reshape(-1, self.hidden_size)
        output = self.fc(lstm_out)

        return output, hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

model = CharLSTM(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(DEVICE)
print(model)

CharLSTM(
  (embedding): Embedding(61, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=61, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(NUM_EPOCHS):
    h = model.init_hidden(BATCH_SIZE, DEVICE)

    for x, y in get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH):
        inputs = torch.from_numpy(x).long().to(DEVICE)
        targets = torch.from_numpy(y).long().to(DEVICE)

        h = tuple([each.data for each in h])

        optimizer.zero_grad()
        output, h = model(inputs, h)
        loss = criterion(output, targets.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

    print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}")

Epoch: 1/20, Loss: 3.2686
Epoch: 2/20, Loss: 3.0475
Epoch: 3/20, Loss: 2.7401
Epoch: 4/20, Loss: 2.5080
Epoch: 5/20, Loss: 2.3642
Epoch: 6/20, Loss: 2.2626


In [ ]:
def predict(model, char, h=None, top_k=None, temperature=1.0):
    model.eval()

    x = np.array([[char2int[char]]])
    input = torch.from_numpy(x).long().to(DEVICE)

    if h is not None:
        h = tuple([each.data for each in h])

    out, h = model(input, h)
    logits = out[-1] / temperature

    p = F.softmax(logits, dim=0).detach().cpu().numpy()

    if top_k is not None:
        p[np.argsort(p)[:-top_k]] = 0
        p = p / p.sum()

    char_index = np.random.choice(len(chars), p=p)

    return int2char[char_index], h

def generate(model, size=500, prime='The ', top_k=5, temperature=1.0):
    model.eval()

    chars_generated = [ch for ch in prime]
    h = model.init_hidden(1, DEVICE)

    for ch in prime:
        char, h = predict(model, ch, h, top_k=top_k, temperature=temperature)

    chars_generated.append(char)

    for _ in range(size - 1):
        char, h = predict(model, chars_generated[-1], h, top_k=top_k, temperature=temperature)
        chars_generated.append(char)

    return ''.join(chars_generated)

In [ ]:
temperatures = [0.25, 0.5, 0.75, 1.0, 1.25]

for temp in temperatures:
    print(f"\nTemperature: {temp}")
    generated_text = generate(model, size=400, prime='KING:', top_k=5, temperature=temp)
    print(generated_text)


Temperature: 0.25
KING:
And the sare the the porest the word the the the the so that the the sand the porest in the courst the the peat the sore the comest and the the some the shall the the condest the peat the the storse the the pratins the sore and the the peace the shand the the sere the condon the the proust the prome to the with the the ward the the condest the coundes the peat the condest the sone the peace the t

Temperature: 0.5
KING::
And the conges, and there to me sare have the pore the coundt the peat hore his a mange to me stord the to selle the sore were will and the seat ties the the warter the stould.

MENENIUS:
I the seard his so the concount a sto the were his the to the could the part the selace the sore hand the preace the sare to the courd seneres the warth the the the could the some and the conter he the courst t

Temperature: 0.75
KING:
This as to he worch to do sell the searter the watring, the with and the some have ward and the shound are sent hat hand the w